In [1]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available()
else 'sin GPU')

True Tesla T4


In [2]:
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 31.3 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92


In [3]:
from roboflow import Roboflow
from google.colab import userdata
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
dataset = project.version(30).download("yolov8")   # ajusta el número de versión al del snippet
print(dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Construction-Site-Safety-30 in yolov8:: 100%|██████████| 1446/1446 [00:00<00:00, 2060.20it/s]


/content/Construction-Site-Safety-30


In [4]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))

In [10]:
import cv2
import os

def extraer_frames_1fps(ruta_video, carpeta_salida):
    # Crear la carpeta de salida si no existe
    if not os.path.exists(carpeta_salida):
        os.makedirs(carpeta_salida)

    # Cargar el video
    video = cv2.VideoCapture(ruta_video)

    # Obtener los FPS originales del video
    fps_original = round(video.get(cv2.CAP_PROP_FPS))
    print(f"El video original está a {fps_original} FPS.")

    conteo_frames = 0
    frames_guardados = 0

    while True:
        exito, frame = video.read()

        if not exito:
            break # Fin del video

        # Guardar solo 1 frame por cada segundo transcurrido
        if conteo_frames % fps_original == 0:
            ruta_frame = os.path.join(carpeta_salida, f"frame_seg_{frames_guardados}.jpg")
            cv2.imwrite(ruta_frame, frame)
            frames_guardados += 1

        conteo_frames += 1

    video.release()
    print(f"Extracción completada. Se guardaron {frames_guardados} frames en la carpeta '{carpeta_salida}'.")

ruta_del_video = "/content/videoplayback 2.mp4"
carpeta_destino = "frames_muestreados"

extraer_frames_1fps(ruta_del_video, carpeta_destino)

El video original está a 30 FPS.
Extracción completada. Se guardaron 14 frames en la carpeta 'frames_muestreados'.


In [6]:
!pip install ultralytics
import ultralytics
ultralytics.checks() # Esto confirma que detecta la GPU T4

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.0/112.6 GB disk)


In [7]:
from ultralytics import YOLO

# Cargar el modelo base pre-entrenado
model_yolo = YOLO('yolov8n.pt')

# Definir la ruta del archivo de configuración del dataset
ruta_yaml = "/content/Construction-Site-Safety-30/data.yaml"

print(f"Iniciando entrenamiento con los datos en: {ruta_yaml}")

# Iniciar el fine-tuning
# 20 épocas es un buen punto de partida para que no tarde una eternidad en la prueba.
resultados_entrenamiento = model_yolo.train(
    data=ruta_yaml,
    epochs=20,
    imgsz=640,       # Tamaño estándar de imagen
    batch=16,        # Tamaño de lote que la T4 maneja bien
    device=0,        # Fuerza el uso de la GPU
    project="Proyecto_UAH",
    name="yolov8n_cascos"
)

Iniciando entrenamiento con los datos en: /content/Construction-Site-Safety-30/data.yaml
Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Construction-Site-Safety-30/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, 

In [11]:
# Cargar el mejor modelo que acaba de entrenarse
mejor_modelo = YOLO('/content/runs/detect/Proyecto_UAH/yolov8n_cascos/weights/best.pt')

resultados_frames = mejor_modelo.predict(source="frames_muestreados/", save=True)


image 1/14 /content/frames_muestreados/frame_seg_0.jpg: 640x384 5 Hardhats, 1 NO-Safety Vest, 10 Persons, 1 Safety Cone, 12 Safety Vests, 75.9ms
image 2/14 /content/frames_muestreados/frame_seg_1.jpg: 640x384 4 Hardhats, 1 NO-Safety Vest, 8 Persons, 12 Safety Vests, 11.5ms
image 3/14 /content/frames_muestreados/frame_seg_10.jpg: 640x384 3 Hardhats, 1 NO-Mask, 5 Persons, 2 Safety Vests, 10.6ms
image 4/14 /content/frames_muestreados/frame_seg_11.jpg: 640x384 3 Hardhats, 2 NO-Safety Vests, 7 Persons, 5 Safety Vests, 10.1ms
image 5/14 /content/frames_muestreados/frame_seg_12.jpg: 640x384 4 Hardhats, 1 NO-Safety Vest, 6 Persons, 8 Safety Vests, 9.7ms
image 6/14 /content/frames_muestreados/frame_seg_13.jpg: 640x384 4 Hardhats, 2 NO-Hardhats, 1 NO-Safety Vest, 8 Persons, 7 Safety Vests, 11.4ms
image 7/14 /content/frames_muestreados/frame_seg_2.jpg: 640x384 9 Hardhats, 1 NO-Hardhat, 1 NO-Safety Vest, 12 Persons, 17 Safety Vests, 9.3ms
image 8/14 /content/frames_muestreados/frame_seg_3.jpg: 64

In [12]:
from ultralytics import YOLO

# 1. Cargar el modelo YOLO-World
model_world = YOLO('yolov8s-world.pt')

# 2. Definir las clases usando texto
clases_a_buscar = ["hard hat", "head", "person"]
model_world.set_classes(clases_a_buscar)

print(f"YOLO-World configurado para buscar: {clases_a_buscar}")

# 3. Ejecutar la predicción
ruta_imagenes = "frames_muestreados/"

resultados_world = model_world.predict(
    source=ruta_imagenes,
    save=True,      # Guarda las imágenes con las cajas dibujadas
    device=0,       # Fuerza el uso de la GPU
    project="Proyecto_UAH",
    name="yolo_world_predicciones"
)

requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 37 packages in 722ms
Prepared 2 packages in 2.33s
Installed 2 packages in 1ms
 + clip==1.0 (from git+https://github.com/ultralytics/CLIP.git@0fa238b2ba553fe76dc158348d8e34625e3e2470)
 + ftfy==6.3.1

requirements: AutoUpdate success ✅ 3.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 197MiB/s]


YOLO-World configurado para buscar: ['hard hat', 'head', 'person']

image 1/14 /content/frames_muestreados/frame_seg_0.jpg: 640x384 3 persons, 55.9ms
image 2/14 /content/frames_muestreados/frame_seg_1.jpg: 640x384 4 persons, 16.3ms
image 3/14 /content/frames_muestreados/frame_seg_10.jpg: 640x384 4 persons, 14.6ms
image 4/14 /content/frames_muestreados/frame_seg_11.jpg: 640x384 4 persons, 14.7ms
image 5/14 /content/frames_muestreados/frame_seg_12.jpg: 640x384 2 persons, 14.5ms
image 6/14 /content/frames_muestreados/frame_seg_13.jpg: 640x384 1 person, 14.3ms
image 7/14 /content/frames_muestreados/frame_seg_2.jpg: 640x384 1 person, 14.2ms
image 8/14 /content/frames_muestreados/frame_seg_3.jpg: 640x384 2 persons, 14.0ms
image 9/14 /content/frames_muestreados/frame_seg_4.jpg: 640x384 (no detections), 13.9ms
image 10/14 /content/frames_muestreados/frame_seg_5.jpg: 640x384 2 persons, 16.4ms
image 11/14 /content/frames_muestreados/frame_seg_6.jpg: 640x384 1 person, 14.4ms
image 12/14 /content/

In [13]:
# VALIDACIÓN mAP@0.5 PARA YOLO-WORLD
print("Validando YOLO-World en el conjunto de test de CSS-Data")
# Usamos el mismo data.yaml que en YOLOv8
metricas_world = model_world.val(data=ruta_yaml, split='test')

# Imprimimos los resultados principales
print(f"mAP@0.5 de YOLO-World: {metricas_world.box.map50:.3f}")

Validando YOLO-World en el conjunto de test de CSS-Data
Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s-world summary: 249 layers, 164,650,217 parameters, 0 gradients, 46.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1493.0±444.5 MB/s, size: 130.9 KB)
val: Scanning /content/Construction-Site-Safety-30/test/labels... 82 images, 8 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 82/82 1.4Kit/s 0.1s
val: New cache created: /content/Construction-Site-Safety-30/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.1it/s 5.3s
                   all         82        806      0.446      0.261      0.279      0.192
             Excavator          7          9      0.419      0.444      0.478       0.33
                Gloves         10         40      0.132      0.475      0.293      0.159
               Hardhat         30        110          1      0.246      0.777   

In [14]:
import torch
import gc

# Eliminar los modelos anteriores de la memoria si siguen cargados
try:
    del model_yolo
    del model_world
except NameError:
    pass

# Vaciar la caché de la GPU
gc.collect()
torch.cuda.empty_cache()
print("Memoria de la GPU liberada. Lista para los VLMs.")

Memoria de la GPU liberada. Lista para los VLMs.


In [15]:
# 1. Instalar dependencias
!pip install -q transformers==4.40.0 einops timm

# 2. Crear un módulo falso para saltar el chequeo de Hugging Face
!mkdir -p /content/flash_attn
!touch /content/flash_attn/__init__.py

import sys
# Asegurar que el entorno lea nuestra carpeta falsa
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

import torch
from transformers import AutoProcessor, AutoModelForCausalLM
from PIL import Image

# 3. Cargar Florence-2
print("Cargando Florence-2...")
modelo_florence = "microsoft/Florence-2-base"
processor_florence = AutoProcessor.from_pretrained(modelo_florence, trust_remote_code=True)
model_florence = AutoModelForCausalLM.from_pretrained(modelo_florence, trust_remote_code=True).to("cuda").eval()

# 4. Preparar imagen y prompt
ruta_imagen = "/content/frames_muestreados/frame_seg_1.jpg"
imagen = Image.open(ruta_imagen).convert("RGB")

tarea = "<OPEN_VOCABULARY_DETECTION>"
prompt_texto = "a person without a hardhat"
prompt_final = tarea + prompt_texto

# 5. Inferencia
print("Ejecutando inferencia...")
inputs = processor_florence(text=prompt_final, images=imagen, return_tensors="pt").to("cuda")
outputs = model_florence.generate(
    input_ids=inputs["input_ids"],
    pixel_values=inputs["pixel_values"],
    max_new_tokens=1024,
    do_sample=False,
    num_beams=3
)

resultado_florence = processor_florence.batch_decode(outputs, skip_special_tokens=False)[0]
respuestas_parseadas = processor_florence.post_process_generation(
    resultado_florence, task=tarea, image_size=(imagen.width, imagen.height)
)

print("Resultados Florence-2:", respuestas_parseadas)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 109.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
Cargando Florence-2...


The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.


preprocessor_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

processing_florence2.py:   0%|          | 0.00/48.7k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base:
- processing_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base:
- configuration_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_florence2.py:   0%|          | 0.00/127k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base:
- modeling_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.


model.safetensors:   0%|          | 0.00/463M [00:00<?, ?B/s]

Ejecutando inferencia...
Resultados Florence-2: {'<OPEN_VOCABULARY_DETECTION>': {'bboxes': [[242.82000732421875, 215.36000061035156, 316.260009765625, 425.2799987792969]], 'bboxes_labels': ['a person without a hardhat'], 'polygons': [], 'polygons_labels': []}}


In [16]:
import torch
import gc
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

# Liberar memoria
try:
    del model_florence
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("Memoria liberada. Florence-2 eliminado de la GPU.")

# Cargar Moondream2
modelo_moon = "vikhyatk/moondream2"
tokenizer_moon = AutoTokenizer.from_pretrained(modelo_moon, revision="2024-05-20")
model_moon = AutoModelForCausalLM.from_pretrained(
    modelo_moon, trust_remote_code=True, revision="2024-05-20"
).to("cuda").eval()

# Inferencia con medición de latencia
pregunta = "¿Hay alguien en la imagen que no lleve casco de seguridad?"
codificado = model_moon.encode_image(imagen)

# Iniciar cronómetro
inicio = time.time()
respuesta_moon = model_moon.answer_question(codificado, pregunta, tokenizer_moon)
fin = time.time()
latencia_vlm = fin - inicio

print(f"Pregunta: {pregunta}")
print(f"Respuesta Moondream2: {respuesta_moon}")
print(f"\nVELOCIDAD VLM: La inferencia tomó {latencia_vlm:.2f} segundos para un (1) frame en GPU T4.")

Memoria liberada. Florence-2 eliminado de la GPU.


tokenizer_config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

...

modeling_phi.py:   0%|          | 0.00/48.0k [00:00<?, ?B/s]

vision_encoder.py:   0%|          | 0.00/10.2k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.73G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Pregunta: ¿Hay alguien en la imagen que no lleve casco de seguridad?
Respuesta Moondream2: In the image, there is a group of construction workers wearing yellow vests and hard hats. One of the workers is wearing a backpack. The workers are working on a construction site, and there is a sign that says " Hispanic Workers." This suggests that the workers are of Hispanic descent, and they are working together on the construction site. The presence of the backpack might indicate that the worker is carrying essential tools or equipment for their job.

VELOCIDAD VLM: La inferencia tomó 5.94 segundos para un (1) frame en GPU T4.


In [17]:
# Definimos 4 variantes del prompt para medir la sensibilidad del VLM
variantes_prompt = [
    "¿Hay alguien en la imagen que no lleve casco de seguridad?", # Original
    "¿Todos los trabajadores están usando casco?", # Variante 1: Pregunta afirmativa con trampa
    "Identifica si alguna persona olvidó ponerse el casco.", # Variante 2: Instrucción directa
    "Are there any construction workers without a hard hat?" # Variante 3: Cambio de idioma
]

print("--- Prueba de Robustez: Sensibilidad al Prompt (Moondream2) ---")
# Reutilizamos la variable 'imagen' y el modelo 'model_moon' que ya tienes cargados en la GPU
codificado = model_moon.encode_image(imagen)

for i, prompt in enumerate(variantes_prompt):
    respuesta = model_moon.answer_question(codificado, prompt, tokenizer_moon)
    print(f"Variante {i+1}: '{prompt}'")
    print(f"Respuesta: {respuesta}\n")

--- Prueba de Robustez: Sensibilidad al Prompt (Moondream2) ---
Variante 1: '¿Hay alguien en la imagen que no lleve casco de seguridad?'
Respuesta: In the image, there is a group of construction workers wearing yellow vests and hard hats. One of the workers is wearing a backpack. The workers are working on a construction site, and there is a sign that says " Hispanic Workers." This suggests that the workers are of Hispanic descent, and they are working together on the construction site. The presence of the backpack might indicate that the worker is carrying essential tools or equipment for their job.

Variante 2: '¿Todos los trabajadores están usando casco?'
Respuesta: Yes, the workers are using a ladder that is made in the United States.

Variante 3: 'Identifica si alguna persona olvidó ponerse el casco.'
Respuesta: Yes

Variante 4: 'Are there any construction workers without a hard hat?'
Respuesta: Yes, there are construction workers without hard hats in the image.



In [21]:
import os
import glob
import torch
import gc
import re
import warnings
from PIL import Image
from ultralytics import YOLO
from transformers import AutoModelForCausalLM, AutoTokenizer

# Silenciar las advertencias rojas de Hugging Face
warnings.filterwarnings("ignore", category=FutureWarning)

def calcular_f1_evento(ground_truth, predicciones):
    tp = sum(1 for gt, pred in zip(ground_truth, predicciones) if gt == 1 and pred == 1)
    fp = sum(1 for gt, pred in zip(ground_truth, predicciones) if gt == 0 and pred == 1)
    fn = sum(1 for gt, pred in zip(ground_truth, predicciones) if gt == 1 and pred == 0)

    denominador = (2 * tp) + fp + fn
    f1_score = (2 * tp) / denominador if denominador > 0 else 0.0
    return f1_score, tp, fp, fn

# --- SOLUCIÓN DEL BUG DE ORDENAMIENTO ---
def extraer_numero(ruta):
    # Extrae el número final del archivo (ej: frame_seg_10.jpg -> 10)
    numeros = re.findall(r'\d+', os.path.basename(ruta))
    return int(numeros[-1]) if numeros else 0

# Ahora los frames se ordenan cronológicamente (0, 1, 2, 3...)
ruta_frames = sorted(glob.glob("frames_muestreados/*.jpg"), key=extraer_numero)

# 1. TU GROUND TRUTH
# 1 = alguien sin casco, 0 = todos con casco
# Nota: Si solo tienes 14 frames guardados, asegúrate de que esta lista tenga 14 elementos.
etiquetas_reales = [1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1]

pred_yolo = []
pred_world = []
pred_vlm = []

print("1. Evaluando YOLOv8n")
mejor_modelo = YOLO('/content/runs/detect/Proyecto_UAH/yolov8n_cascos/weights/best.pt')
for frame_path in ruta_frames:
    res_yolo = mejor_modelo(frame_path, verbose=False)[0]
    clases_yolo = [res_yolo.names[int(c)] for c in res_yolo.boxes.cls]
    pred_yolo.append(1 if "NO-Hardhat" in clases_yolo else 0)

del mejor_modelo
gc.collect()
torch.cuda.empty_cache()

print("2. Evaluando YOLO-World")
model_world = YOLO('yolov8s-world.pt')
model_world.set_classes(["hard hat", "head", "person"])
for frame_path in ruta_frames:
    res_world = model_world(frame_path, verbose=False)[0]
    clases_world = [res_world.names[int(c)] for c in res_world.boxes.cls]
    pred_world.append(1 if "head" in clases_world else 0)

del model_world
gc.collect()
torch.cuda.empty_cache()

print("3. Evaluando Moondream2")
modelo_moon_name = "vikhyatk/moondream2"
tokenizer_moon = AutoTokenizer.from_pretrained(modelo_moon_name, revision="2024-05-20")
model_moon = AutoModelForCausalLM.from_pretrained(
    modelo_moon_name, trust_remote_code=True, revision="2024-05-20"
).to("cuda").eval()

for frame_path in ruta_frames:
    img_vlm = Image.open(frame_path).convert("RGB")
    codificado = model_moon.encode_image(img_vlm)
    ans_vlm = model_moon.answer_question(
        codificado,
        "¿Hay alguien en la imagen que no lleve casco? Responde solo 'Yes' o 'No'.",
        tokenizer_moon
    )
    pred_vlm.append(1 if "yes" in ans_vlm.lower() else 0)

del model_moon
gc.collect()
torch.cuda.empty_cache()

print("\n--- PREDICCIONES GENERADAS ---")
print(f"Realidad: {etiquetas_reales}")
print(f"YOLOv8  : {pred_yolo}")
print(f"Y-World : {pred_world}")
print(f"VLM     : {pred_vlm}")

modelos = {
    "YOLOv8n (Cerrado)": pred_yolo,
    "YOLO-World (Abierto)": pred_world,
    "Moondream2 (VLM)": pred_vlm
}

print("\n=== RESULTADOS DE PRECISIÓN NIVEL-EVENTO (F1) ===")
for nombre_modelo, predicciones in modelos.items():
    min_len = min(len(etiquetas_reales), len(predicciones))
    f1, tp, fp, fn = calcular_f1_evento(etiquetas_reales[:min_len], predicciones[:min_len])

    print(f"\n{nombre_modelo}:")
    print(f"  TP: {tp} | FP: {fp} | FN: {fn}")
    print(f"  >> Score F1 : {f1:.3f}")

1. Evaluando YOLOv8n
2. Evaluando YOLO-World
3. Evaluando Moondream2


The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.



--- PREDICCIONES GENERADAS ---
Realidad: [1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1]
YOLOv8  : [0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1]
Y-World : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
VLM     : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

=== RESULTADOS DE PRECISIÓN NIVEL-EVENTO (F1) ===

YOLOv8n (Cerrado):
  TP: 2 | FP: 1 | FN: 4
  >> Score F1 : 0.444

YOLO-World (Abierto):
  TP: 0 | FP: 0 | FN: 6
  >> Score F1 : 0.000

Moondream2 (VLM):
  TP: 6 | FP: 8 | FN: 0
  >> Score F1 : 0.600
